In [3]:
# 1. IMPORT LIBRARIES

import pandas as pd
import numpy as np

# 2. LOAD DATA

features = pd.read_csv('../data/raw/ecommerce_customer_features.csv')
targets = pd.read_csv('../data/raw/ecommerce_customer_targets.csv')

print("Features shape:", features.shape)
print("Targets shape:", targets.shape)

# 3. MERGE DATA
#
df = pd.concat([features, targets], axis=1)

print("Merged shape:", df.shape)
df.head()

Features shape: (6000, 15)
Targets shape: (6000, 2)
Merged shape: (6000, 17)


,Customer_ID,account_age_months,avg_order_value,total_orders,days_since_last_purchase,discount_usage_rate,return_rate,customer_support_tickets,loyalty_member,browsing_frequency_per_week,cart_abandonment_rate,product_review_score_avg,engagement_score,satisfaction_score,price_sensitivity_index,Customer_ID,churned
0,0520df14-712d-4c69-a0c5-95a2e7dfc1ff,46,164.96,12,17,0.243,0.1720,0,No,6.1,0.430,5.00,6.58,9.43,3.7,0520df14-712d-4c69-a0c5-95a2e7dfc1ff,No
1,a4013b3f-0688-4096-a194-6074be8ffec8,3,39.09,4,5,0.591,0.0808,1,No,4.1,0.183,4.44,6.25,8.50,6.9,a4013b3f-0688-4096-a194-6074be8ffec8,No
2,eb870f2c-ed3d-4a21-a8ac-273fae69ea4f,29,37.42,8,47,0.212,0.1424,0,No,1.2,0.426,3.87,3.32,8.40,4.3,eb870f2c-ed3d-4a21-a8ac-273fae69ea4f,No
3,a7433451-8ea9-428a-9d80-679c6963b39f,35,62.64,9,3,0.699,0.0128,0,No,3.8,0.730,4.75,6.42,9.71,7.5,a7433451-8ea9-428a-9d80-679c6963b39f,No
4,43f81935-49e3-44d3-94d1-5c4715738988,39,113.03,1,7,0.382,0.0232,0,No,5.4,0.613,5.00,6.48,9.92,5.0,43f81935-49e3-44d3-94d1-5c4715738988,No


In [4]:
# 4. DATA CLEANING

# Check missing values
print(df.isnull().sum())

# Fill missing values
df.fillna(method='ffill', inplace=True)

# Remove duplicates
df.drop_duplicates(inplace=True)

print("Cleaned shape:", df.shape)

Customer_ID                    0
account_age_months             0
avg_order_value                0
total_orders                   0
days_since_last_purchase       0
discount_usage_rate            0
return_rate                    0
customer_support_tickets       0
loyalty_member                 0
browsing_frequency_per_week    0
cart_abandonment_rate          0
product_review_score_avg       0
engagement_score               0
satisfaction_score             0
price_sensitivity_index        0
Customer_ID                    0
churned                        0
dtype: int64
Cleaned shape: (6000, 17)


C:\Users\ASUS\AppData\Local\Temp\ipykernel_22304\3234436530.py:7: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df.fillna(method='ffill', inplace=True)


In [5]:
print(targets.head())
print(targets.columns)

                            Customer_ID churned
0  0520df14-712d-4c69-a0c5-95a2e7dfc1ff      No
1  a4013b3f-0688-4096-a194-6074be8ffec8      No
2  eb870f2c-ed3d-4a21-a8ac-273fae69ea4f      No
3  a7433451-8ea9-428a-9d80-679c6963b39f      No
4  43f81935-49e3-44d3-94d1-5c4715738988      No
Index(['Customer_ID', 'churned'], dtype='object')


In [6]:
targets = targets.drop(columns=['Customer_ID'])

In [7]:
# 6. Mapping correction

df = features.copy()
df['Churn'] = targets['churned']

df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})

print(df.head())
print(df['Churn'].value_counts())

                            Customer_ID  account_age_months  avg_order_value  \
0  0520df14-712d-4c69-a0c5-95a2e7dfc1ff                  46           164.96   
1  a4013b3f-0688-4096-a194-6074be8ffec8                   3            39.09   
2  eb870f2c-ed3d-4a21-a8ac-273fae69ea4f                  29            37.42   
3  a7433451-8ea9-428a-9d80-679c6963b39f                  35            62.64   
4  43f81935-49e3-44d3-94d1-5c4715738988                  39           113.03   

   total_orders  days_since_last_purchase  discount_usage_rate  return_rate  \
0            12                        17                0.243       0.1720   
1             4                         5                0.591       0.0808   
2             8                        47                0.212       0.1424   
3             9                         3                0.699       0.0128   
4             1                         7                0.382       0.0232   

   customer_support_tickets loyalty_member  

In [8]:
# 5. ENCODING

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in df.select_dtypes(include='object').columns:
    if col != 'Churn':   # DO NOT encode target
        df[col] = le.fit_transform(df[col])

In [9]:

# 7. MODEL TRAINING + EVALUATION

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


if 'Customer_ID' in df.columns:
    df = df.drop(columns=['Customer_ID'])


X = df.drop('Churn', axis=1)
y = df['Churn']
feature_columns = X.columns

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


# TRAIN MODEL

model = RandomForestClassifier(
    random_state=42,
    class_weight='balanced'   
)

model.fit(X_train, y_train)


# DEFAULT PREDICTION

preds = model.predict(X_test)

print("===== DEFAULT MODEL =====")
print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))


# THRESHOLD TUNING (IMPROVE RECALL)

probs = model.predict_proba(X_test)[:, 1]

threshold = 0.4   # try 0.45, 0.4, 0.35
preds_new = (probs > threshold).astype(int)

print("\n===== AFTER THRESHOLD TUNING =====")
print("Threshold:", threshold)
print("Accuracy:", accuracy_score(y_test, preds_new))
print(classification_report(y_test, preds_new))




# FEATURE IMPORTANCE

importance = pd.Series(model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print("\n===== TOP FEATURES =====")
print(importance.head(10))

===== DEFAULT MODEL =====
Accuracy: 0.9708333333333333
              precision    recall  f1-score   support

           0       0.97      0.99      0.98      1007
           1       0.95      0.87      0.91       193

    accuracy                           0.97      1200
   macro avg       0.96      0.93      0.94      1200
weighted avg       0.97      0.97      0.97      1200


===== AFTER THRESHOLD TUNING =====
Threshold: 0.4
Accuracy: 0.9683333333333334
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      1007
           1       0.91      0.89      0.90       193

    accuracy                           0.97      1200
   macro avg       0.94      0.94      0.94      1200
weighted avg       0.97      0.97      0.97      1200


===== TOP FEATURES =====
engagement_score               0.434092
days_since_last_purchase       0.392209
satisfaction_score             0.032009
browsing_frequency_per_week    0.020160
product_review_score_avg 

In [10]:
import pickle

pickle.dump(feature_columns, open("features.pkl", "wb"))

In [11]:
pickle.dump(model, open("model.pkl", "wb"))